In [1]:
import re
from pytube import YouTube
from langchain_classic.tools import tool
from IPython.display import display, JSON
import yt_dlp
from typing import List, Dict
from langchain_core.messages import HumanMessage, ToolMessage
import json

import warnings
warnings.filterwarnings("ignore")

import logging
pytube_logger = logging.getLogger('pytube')
pytube_logger.setLevel(logging.ERROR)

yt_dlp_logger = logging.getLogger('yt_dlp')
yt_dlp_logger.setLevel(logging.ERROR)

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
from langchain.chat_models import init_chat_model

llm = init_chat_model("claude-haiku-4-5-20251001",model_provider="anthropic")

#### Video ID extraction tool

In [4]:
from urllib.parse import urlparse, parse_qs

In [5]:
@tool
def extract_video_id(url: str) -> str:
    """
    Extracts the 11-character YouTube video ID from a URL.
    
    Args:
        url (str): A YouTube URL containing a video ID.

    Returns:
        str: Extracted video ID or error message if parsing fails.
    """

    parsed_url = urlparse(url)

    if parsed_url.hostname in ["www.youtube.com", "youtube.com", "m.youtube.com"]:
        if parsed_url.path == "/watch":
            return parse_qs(parsed_url.query).get("v", [None])[0]

        # https://youtube.com/shorts/VIDEO_ID
        if parsed_url.path.startswith("/shorts/"):
            return parsed_url.path.split("/")[2]

        # https://youtube.com/embed/VIDEO_ID
        if parsed_url.path.startswith("/embed/"):
            return parsed_url.path.split("/")[2]

    # https://youtu.be/VIDEO_ID
    if parsed_url.hostname == "youtu.be":
        return parsed_url.path.lstrip("/")

    return None

In [6]:
print(extract_video_id.name)
print("----------------------------")
print(extract_video_id.description)
print("----------------------------")
print(extract_video_id.func)

extract_video_id
----------------------------
Extracts the 11-character YouTube video ID from a URL.

Args:
    url (str): A YouTube URL containing a video ID.

Returns:
    str: Extracted video ID or error message if parsing fails.
----------------------------
<function extract_video_id at 0x00000215BEC7E5C0>


In [7]:
extract_video_id.run("https://www.youtube.com/watch?v=hfIUstzHs9A")

'hfIUstzHs9A'

In [8]:
extract_video_id

StructuredTool(name='extract_video_id', description='Extracts the 11-character YouTube video ID from a URL.\n\nArgs:\n    url (str): A YouTube URL containing a video ID.\n\nReturns:\n    str: Extracted video ID or error message if parsing fails.', args_schema=<class 'langchain_core.utils.pydantic.extract_video_id'>, func=<function extract_video_id at 0x00000215BEC7E5C0>)

#### Tool list

In [9]:
tools = []
tools.append(extract_video_id)

#### Transcript Fetching Tool

In [10]:
from youtube_transcript_api import YouTubeTranscriptApi

@tool
def fetch_transcript(video_id: str, language: str = "en") -> str:
    """
    Fetches the transcript of a YouTube video.

    Args:
        video_id (str) : The YouTube video ID (e.g., "dQw4w9WgXcQ").
        language (str) : Language code for the transcript (e.g., "en", "es").
    
    Returns:
        str: The transcript text or an error message.
    """

    try:
        ytt_api = YouTubeTranscriptApi()
        transcript = ytt_api.fetch(video_id, languages=[language])
        return " ".join([snippet.text for snippet in transcript.snippets])
    except Exception as e:
        return f"Error: {str(e)}"

In [11]:
fetch_transcript.run("hfIUstzHs9A")

'Over the past couple of months, large language models, or LLMs, such as chatGPT, have taken the world by storm. Whether it\'s writing poetry or helping plan your upcoming vacation, we are seeing a step change in the performance of AI and its potential to drive enterprise value. My name is Kate Soule. I\'m a senior manager of business strategy at IBM Research, and today I\'m going to give a brief overview of this new field of AI that\'s emerging and how it can be used in a business setting to drive value. Now, large language models are actually a part of a different class of models called foundation models. Now, the term "foundation models" was actually first coined by a team from Stanford when they saw that the field of AI was converging to a new paradigm. Where before AI applications were being built by training, maybe a library of different AI models, where each AI model was trained on very task-specific data to perform very specific task. They predicted that we were going to start 

In [12]:
tools.append(fetch_transcript)

#### YouTube Search Tool

In [13]:
from pytubefix import Search

@tool
def search_youtube(query: str) -> List[Dict[str, str]]:
    """
    Search YouTube for videos matching the query.

    Args:
        query (str): The search term to look for on YouTube

    Returns:
        List of dictionaries containing video titles and IDs in format:
        [{'title': 'Video Title', 'video_id': 'abc123'}, ...]
        Returns error message if search fails
    """
    try:
        s = Search(query)
        return [
            {
                "title": yt.title,
                "video_id": yt.video_id,
                "url": f"https://youtu.be/{yt.video_id}"
            }
            for yt in s.results
        ]
    except Exception as e:
        return f"Error: {str(e)}"

In [14]:
search_out = search_youtube.run("Generative AI")


In [15]:
tools.append(search_youtube)

#### Metadata Extraction Tool

In [16]:
@tool
def get_full_metadata(url: str) -> dict:
    """Extract metadata given a youtube URL, including title, views, duration, channel, likes, comments, and chapters."""
    with yt_dlp.YoutubeDL({'quiet': True, 'logger': yt_dlp_logger}) as ydl:
        info = ydl.extract_info(url, download=False)
        return{
            'title': info.get('title'),
            'views': info.get('view_count'),
            'duration': info.get('duration'),
            'channel': info.get('uploader'),
            'likes': info.get('like_count'),
            'comments': info.get('comment_count'),
            'chapters': info.get('chapters',[])
        }

In [17]:
meta_data=get_full_metadata.run("https://www.youtube.com/watch?v=T-D1OfcDW1M")
display(JSON(meta_data))

<IPython.core.display.JSON object>

In [18]:
tools.append(get_full_metadata)

#### Thumbnail Retrieval Tool

In [19]:
@tool
def get_thumbnails(url:str) -> List[Dict]:
    """
    Get available thumbnails for a youtube video using its URL.

    Args:
        url (str): YouTube video URL (any format)

    Returns:
        List of dictionaries with thumbnail URLs and resolutions in YouTube's native order.
    """

    try:
        with yt_dlp.YoutubeDL({'quiet': True, 'logger': yt_dlp_logger}) as ydl:
            info = ydl.extract_info(url, download=False)

            thumbnails = []
            for t in info.get('thumbnails',[]):
                if 'url' in t:
                    thumbnails.append({
                        "url": t['url'],
                        "width": t.get('width'),
                        "height": t.get('height'),
                        "resolution": f"{t.get('width','')}x{t.get('height','')}".strip('x')
                    })
            return thumbnails
    except Exception as e:
        return [{"error": f"Failed to get thumbnails: {str(e)}"}]

In [20]:
thumbnails=get_thumbnails.run("https://www.youtube.com/watch?v=qWHaMrR5WHQ")



In [21]:
tools.append(get_thumbnails)

#### Binding Tools

In [22]:
llm_with_tools = llm.bind_tools(tools)

- Tool Schema

In [23]:
for tool in tools:
    schema = {
        "name": tool.name,
        "description": tool.description,
        "parameters": tool.args_schema.schema() if tool.args_schema else {},
        "return": tool.return_type if hasattr(tool, "return_type") else None
    }
    display(JSON(schema))

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

In [24]:
query = "I want to summarize youtube video: https://www.youtube.com/watch?v=T-D1OfcDW1M in english"
print(query)

I want to summarize youtube video: https://www.youtube.com/watch?v=T-D1OfcDW1M in english


In [25]:
messages = [HumanMessage(content=query)]
print(messages)

[HumanMessage(content='I want to summarize youtube video: https://www.youtube.com/watch?v=T-D1OfcDW1M in english', additional_kwargs={}, response_metadata={})]


In [26]:
response_1 = llm_with_tools.invoke(messages)
response_1

AIMessage(content=[{'text': "I'll extract the video ID from the URL and fetch the transcript for you.", 'type': 'text'}, {'id': 'toolu_011ELaSTqUUWn7GGBUdxj2F4', 'caller': {'type': 'direct'}, 'input': {'url': 'https://www.youtube.com/watch?v=T-D1OfcDW1M'}, 'name': 'extract_video_id', 'type': 'tool_use', 'toolset_name': None}], additional_kwargs={}, response_metadata={'id': 'msg_011CfTjoTSdHG2ghUGGzdb3S', 'container': None, 'model': 'claude-haiku-4-5-20251001', 'stop_details': None, 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'not_available', 'input_tokens': 1107, 'output_tokens': 94, 'output_tokens_details': None, 'server_tool_use': None, 'service_tier': 'standard'}, 'diagnostics': None, 'model_name': 'claude-haiku-4-5-20251001', 'model_provider': 'anthropic'}, id='lc_run--01a0e247-71cd-7801-ab1e-ef8c409d0d2

In [27]:
messages.append(response_1)

- Extracting tool call information

In [28]:
tool_mapping = {
    "get_thumbnails": get_thumbnails,
    "extract_video_id": extract_video_id,
    "fetch_transcript": fetch_transcript,
    "search_youtube": search_youtube,
    "get_full_metadata": get_full_metadata
}

In [29]:
from pprint import pprint
tool_calls_1 = response_1.tool_calls
pprint(tool_calls_1)

[{'args': {'url': 'https://www.youtube.com/watch?v=T-D1OfcDW1M'},
  'id': 'toolu_011ELaSTqUUWn7GGBUdxj2F4',
  'name': 'extract_video_id',
  'type': 'tool_call'}]


In [30]:
tool_name = tool_calls_1[0]['name']
print(tool_name)

extract_video_id


In [31]:
tool_call_id =tool_calls_1[0]['id']
print(tool_call_id)

toolu_011ELaSTqUUWn7GGBUdxj2F4


In [32]:
args=tool_calls_1[0]['args']
print(args)

{'url': 'https://www.youtube.com/watch?v=T-D1OfcDW1M'}


In [33]:
my_tool = tool_mapping[tool_calls_1[0]['name']]

In [34]:
video_id = my_tool.invoke(tool_calls_1[0]['args'])
video_id

'T-D1OfcDW1M'

In [35]:
messages.append(ToolMessage(content=video_id, tool_call_id = tool_calls_1[0]['id']))

In [36]:
response_2 = llm_with_tools.invoke(messages)
response_2

AIMessage(content=[{'text': 'Now let me fetch the transcript:', 'type': 'text'}, {'id': 'toolu_01QPzGDsFALu51Cax3usqS3u', 'caller': {'type': 'direct'}, 'input': {'video_id': 'T-D1OfcDW1M', 'language': 'en'}, 'name': 'fetch_transcript', 'type': 'tool_use', 'toolset_name': None}], additional_kwargs={}, response_metadata={'id': 'msg_011CfTjoaVHfBKJS6revYVB9', 'container': None, 'model': 'claude-haiku-4-5-20251001', 'stop_details': None, 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'not_available', 'input_tokens': 1224, 'output_tokens': 90, 'output_tokens_details': None, 'server_tool_use': None, 'service_tier': 'standard'}, 'diagnostics': None, 'model_name': 'claude-haiku-4-5-20251001', 'model_provider': 'anthropic'}, id='lc_run--01a0e247-7a07-75a0-80c2-ee95deccc0ad-0', tool_calls=[{'name': 'fetch_transcript', 'a

In [37]:
messages.append(response_2)

In [38]:
tool_calls_2 = response_2.tool_calls
tool_calls_2

[{'name': 'fetch_transcript',
  'args': {'video_id': 'T-D1OfcDW1M', 'language': 'en'},
  'id': 'toolu_01QPzGDsFALu51Cax3usqS3u',
  'type': 'tool_call'}]

In [39]:
fetch_transcript_tool_output = tool_mapping[tool_calls_2[0]['name']].invoke(tool_calls_2[0]['args'])
fetch_transcript_tool_output

'Large language models. They are everywhere. They get some things amazingly right and other things very interestingly wrong. My name\xa0is Marina Danilevsky. I am a Senior Research Scientist here at IBM Research. And I want\xa0to tell you about a framework to help large language models be more accurate and more up to\xa0date: Retrieval-Augmented Generation, or RAG. Let\'s just talk about the "Generation" part for a\xa0minute. So forget the "Retrieval-Augmented". So the\xa0generation, this refers to large language models,\xa0or LLMs, that generate text in response to a user query, referred to as a prompt. These\xa0models can have some undesirable behavior. I want to tell you an anecdote to illustrate this. So my kids, they recently asked me this question: "In our solar system, what planet has the most\xa0moons?" And my response was, “Oh, that\'s really great that you\'re asking this question. I loved\xa0space when I was your age.” Of course, that was like 30 years ago. But I know this! 

In [40]:
messages.append(ToolMessage(content = fetch_transcript_tool_output, tool_call_id = tool_calls_2[0]['id']))

In [41]:
summary = llm_with_tools.invoke(messages)
summary

AIMessage(content='## Summary\n\nThis video is about **Retrieval-Augmented Generation (RAG)**, a framework designed to improve the accuracy and currency of large language models (LLMs).\n\n**Key Points:**\n\n1. **The Problem with LLMs**: LLMs can confidently provide incorrect or outdated information because they rely solely on data from their training. The speaker illustrates this with an anecdote about answering a question about which planet has the most moons - she initially said Jupiter (outdated info), when the correct answer is Saturn.\n\n2. **Two Main Issues**:\n   - **Hallucination**: LLMs make up answers without credible sources\n   - **Outdated Information**: Models can\'t keep up with new discoveries or changes\n\n3. **How RAG Works**:\n   - Instead of relying only on what the LLM knows, RAG adds a content store (internet, documents, or policies)\n   - The LLM first retrieves relevant information from this content store\n   - It then combines the retrieved content with the us

#### Automating the tool calling process

In [42]:
# Define the processing steps
def execute_tool(tool_call):
    """Execute single tool call and return ToolMessage"""

    try:
        result = tool_mapping[tool_call['name']].invoke(tool_call['args'])
        return ToolMessage(
            content=str(result),
            tool_call_id = tool_call['id']
        )
    except Exception as e:
        return ToolMessage(
            content=f"Error: {str(e)}",
            tool_call_id = tool_call['id']
        )

In [43]:
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

#### Summarization Chain

In [44]:
summarization_chain = (
    # Start with initial query
    RunnablePassthrough.assign(
        messages=lambda x: [HumanMessage(content=x["query"])]
    )
    # First LLm call (extract video ID)
    | RunnablePassthrough.assign(
        ai_response=lambda x: llm_with_tools.invoke(x["messages"])
    )
    # Process first tool call
    | RunnablePassthrough.assign(
        tool_messages = lambda x: [
            execute_tool(tc) for tc in x["ai_response"].tool_calls
        ]
    )
    # Update Message history
    | RunnablePassthrough.assign(
        messages=lambda x: x["messages"] + [x["ai_response"]] + x["tool_messages"]
    )
    # Second LLM call (fetch transcript)
    | RunnablePassthrough.assign(
        ai_response2=lambda x: llm_with_tools.invoke(x["messages"])
    )
    # Process second tool call
    | RunnablePassthrough.assign(
        tool_messages2=lambda x: [
            execute_tool(tc) for tc in x["ai_response2"].tool_calls
        ]
    )
    # Final message update
    | RunnablePassthrough.assign(
        messages=lambda x: x["messages"] + [x["ai_response2"]] + x["tool_messages2"]
    )
    # Generate Final Summary
    | RunnablePassthrough.assign(
        summary=lambda x: llm_with_tools.invoke(x["messages"]).content
    )
    # Return just the summary text
    | RunnableLambda(lambda x: x["summary"])
)

In [45]:
result = summarization_chain.invoke({
    "query": "Summarize this YouTube video: https://www.youtube.com/watch?v=1bUy-1hGZpI"
})

print("Video Summary:\n", result)

Video Summary:
 ## Summary of the Video

This video is an introduction to **LangChain**, an open-source orchestration framework for building applications with large language models (LLMs).

### Key Points:

**What is LangChain?**
- An open-source framework for developing LLM applications, available in Python and JavaScript
- Provides a centralized development environment and generic interface for any LLM
- Launched by Harrison Chase in October 2022 and became GitHub's fastest-growing open-source project by June 2023

**Core Components:**
1. **LLM Module** - Standard interface for different models (GPT-4, Llama 2, etc.)
2. **Prompts** - Instructions to language models, including templates, few-shot examples, and output formatting
3. **Chains** - Sequential execution of functions where output feeds into the next step
4. **Indexes** - Ways to access external data (document loaders, vector databases, text splitters)
5. **Memory** - Utilities to give applications long-term conversation hist

In [46]:
query = {"query": "Get top 3 youtube videos in India and their metadata"}
try:
    result = summarization_chain.invoke(query)
    print("Video Summary:\n", result)
except Exception as e:
    print("Non-critical network error:", e)

Video Summary:
 Perfect! Here are the **top 3 YouTube videos in India** with their metadata:

---

### **1. Udta Teer Official Trailer**
- **Title:** Udta Teer Official Trailer | Ayushmann Khurrana & Sara Ali Khan | Aakash Kaushik | In Cinemas Oct 9
- **Channel:** Dharma Productions
- **Views:** 32,991,906
- **Likes:** 149,972
- **Comments:** 16,000
- **Duration:** 199 seconds (~3 minutes 19 seconds)
- **Video ID:** hN_ZUElLH44

---

### **2. Jaanat ने Abhishek को रंगे हाथों पकड़ा**
- **Title:** Jaanat ने Abhishek को रंगे हाथों पकड़ा | Indian Game Show | Top Moments
- **Channel:** SET India
- **Views:** 1,339,059
- **Likes:** 10,189
- **Comments:** 186
- **Duration:** 1,094 seconds (~18 minutes 14 seconds)
- **Video ID:** 00HPdjUiyqI

---

### **3. Harssh के तोते उड़ाने को तैयार हुई Bharti**
- **Title:** Harssh के तोते उड़ाने को तैयार हुई Bharti | Indian Game Show | Best Comedy
- **Channel:** SET India
- **Views:** 782,591
- **Likes:** 5,414
- **Comments:** 72
- **Duration:** 1,070 sec

#### Recursive Chain flow

In [47]:
from langchain_core.runnables import RunnableBranch, RunnableLambda
from langchain_core.messages import HumanMessage, ToolMessage
import json

def execute_tool(tool_call):
    """Execute single tool call and return ToolMessage"""
    try:
        result = tool_mapping[tool_call['name']].invoke(tool_call["args"])
        content = json.dumps(result) if isinstance(result, (dict, list)) else str(result)
    except Exception as e:
        content = f"Error: {str(e)}"

    return ToolMessage(
        content=content,
        tool_call_id = tool_call["id"]
    )

In [48]:
# Core processing logic
def process_tool_calls(messages):
    """Recursive tool call processor"""
    last_message = messages[-1]

    # Execute all tool calls in parallel
    tool_messages = [
        execute_tool(tc)
        for tc in getattr(last_message, 'tool_calls', [])
    ]
    # Add tool responses to message history
    updated_messages = messages + tool_messages

    # Get next LLM response
    next_ai_response = llm_with_tools.invoke(updated_messages)
    return updated_messages + [next_ai_response] 

In [49]:
# Recursive stopping condition
def should_continue(messages):
    """Check if you need another iteration"""
    last_message = messages[-1]
    return bool(getattr(last_message, 'tool_calls', None))

In [50]:
# Implementing the Recursive Function
def _recursive_chain(messages):
    """Recursively process tool calls until completion"""
    if should_continue(messages):
        new_messages = process_tool_calls(messages)
        return _recursive_chain(new_messages)
    return messages

recursive_chain = RunnableLambda(_recursive_chain)

#### Universal Chain

In [51]:
universal_chain = (
    RunnableLambda(lambda x: [HumanMessage(content=x["query"])])
    | RunnableLambda(lambda messages: messages +[llm_with_tools.invoke(messages)])
    | recursive_chain
)

In [52]:
query_us = {"query": "Show top 3 US trending videos with metadata and thumbnails"}

try:
    response = universal_chain.invoke(query_us)
    print("\nUS Trending Videos:\n", response[-1])
except Exception as e:
    print("Non-critical network error while fetching US trending videos:", e)


US Trending Videos:
 content='Perfect! Here are the **Top 3 US Trending Videos** with metadata and thumbnails:\n\n---\n\n## 🎥 **Video #1: "5 USA Faceless Niches Nobody Talks About"**\n**Channel:** Abhi AI  \n**Video ID:** XpbymPeiAwY\n\n### Metadata:\n- **Views:** 13,694\n- **Duration:** 4 minutes 36 seconds\n- **Likes:** Not available\n- **Comments:** 65\n- **Chapters:** No chapters available\n\n### Thumbnail (High Quality):\n![Video 1 Thumbnail](https://i.ytimg.com/vi/XpbymPeiAwY/maxresdefault.jpg)\n\n---\n\n## 🎥 **Video #2: "Top 12 High RPM USA YouTube Niches for 2026 (Faceless + AI)"** ⭐ **Most Popular**\n**Channel:** Rebelvids  \n**Video ID:** tmaV47vwnjM\n\n### Metadata:\n- **Views:** 77,698 👑\n- **Duration:** 9 minutes 36 seconds\n- **Likes:** 2,226\n- **Comments:** 134\n- **Chapters:** 13 chapters including:\n  - Pet Psychology\n  - Human Psychology\n  - Space & Universe\n  - Military Content\n  - Real Estate\n  - True Horror Stories\n  - And 7 more...\n\n### Thumbnail (High Q

In [53]:
tools

[StructuredTool(name='extract_video_id', description='Extracts the 11-character YouTube video ID from a URL.\n\nArgs:\n    url (str): A YouTube URL containing a video ID.\n\nReturns:\n    str: Extracted video ID or error message if parsing fails.', args_schema=<class 'langchain_core.utils.pydantic.extract_video_id'>, func=<function extract_video_id at 0x00000215BEC7E5C0>),
 StructuredTool(name='fetch_transcript', description='Fetches the transcript of a YouTube video.\n\nArgs:\n    video_id (str) : The YouTube video ID (e.g., "dQw4w9WgXcQ").\n    language (str) : Language code for the transcript (e.g., "en", "es").\n\nReturns:\n    str: The transcript text or an error message.', args_schema=<class 'langchain_core.utils.pydantic.fetch_transcript'>, func=<function fetch_transcript at 0x00000215C1105080>),
 StructuredTool(name='search_youtube', description="Search YouTube for videos matching the query.\n\nArgs:\n    query (str): The search term to look for on YouTube\n\nReturns:\n    List

In [54]:
url = "https://youtu.be/W_XXHnuxq9I?si=JSdqtJi4pLnyy5_L"

In [55]:
video_id = extract_video_id.run(url)
print(f"Extracted video ID: {video_id}")

Extracted video ID: W_XXHnuxq9I


In [56]:
video_metadata= get_full_metadata.run(url)
pprint(video_metadata)

{'channel': 'Euron',
 'chapters': None,
 'comments': 3,
 'duration': 7076,
 'likes': 16,
 'title': 'Every Real AI Project Starts Here: How to Create a Client Proposal '
          '| Super30 FDE Part 2',
 'views': 2531}


In [57]:
transcript = fetch_transcript.run("W_XXHnuxq9I")
pprint(transcript)

("Hi everyone. I believe I'm visible to all of you guys. So please confirm so "
 'that we can start. Okay. So good afternoon everyone. So yesterday I was '
 'talking about this uh project and we have discussed about the entire client '
 "requirement. So now today what we are going to do? So we'll [clears throat] "
 "try to work on the proposal side and we'll try to understand that what and "
 'all things will go inside the proposal and then if time allows then in that '
 "case we'll try to uh even understand that uh yeah proposal we are going to "
 "discuss that's uh one part second part is so if time allows then we can try "
 'to uh discuss uh capacity planning as well capacity planning and costing and '
 "then HLD and LLD we'll be talking out. So, let me share my screen and yeah, "
 "let's start. [clears throat] Okay. So fine guys my student is visible right "
 "everyone can you please confirm quickly? Yes. Okay fine cool. So let's start "
 'with the proposal document. So yesterday w

In [58]:
thumbnails = get_thumbnails.run(url)
print(thumbnails)

[{'url': 'https://i.ytimg.com/vi/W_XXHnuxq9I/3.jpg', 'width': None, 'height': None, 'resolution': ''}, {'url': 'https://i.ytimg.com/vi_webp/W_XXHnuxq9I/3.webp', 'width': None, 'height': None, 'resolution': ''}, {'url': 'https://i.ytimg.com/vi/W_XXHnuxq9I/2.jpg', 'width': None, 'height': None, 'resolution': ''}, {'url': 'https://i.ytimg.com/vi_webp/W_XXHnuxq9I/2.webp', 'width': None, 'height': None, 'resolution': ''}, {'url': 'https://i.ytimg.com/vi/W_XXHnuxq9I/1.jpg', 'width': None, 'height': None, 'resolution': ''}, {'url': 'https://i.ytimg.com/vi_webp/W_XXHnuxq9I/1.webp', 'width': None, 'height': None, 'resolution': ''}, {'url': 'https://i.ytimg.com/vi/W_XXHnuxq9I/mq3.jpg', 'width': None, 'height': None, 'resolution': ''}, {'url': 'https://i.ytimg.com/vi_webp/W_XXHnuxq9I/mq3.webp', 'width': None, 'height': None, 'resolution': ''}, {'url': 'https://i.ytimg.com/vi/W_XXHnuxq9I/mq2.jpg', 'width': None, 'height': None, 'resolution': ''}, {'url': 'https://i.ytimg.com/vi_webp/W_XXHnuxq9I/mq

In [59]:
prompt = f"""
Please analyze this YouTube video and provide a comprehensive summary.

VIDEO TITLE: {video_metadata['title']}
CHANNEL: {video_metadata['channel']}
VIEWS: {video_metadata['views']}
DURATION: {video_metadata['duration']} seconds
LIKES: {video_metadata['likes']}

TRANSCRIPT EXCERPT:
{transcript[:3000]}... (transcript truncated for brevity)

Based on this information, please provide:
1. A concise summary of the video content (3-5 bullet points)
2. The main topics or themes discussed
3. The intended audience for this content
4. A brief analysis of why this video might be performing well (or not)
"""

In [60]:
messages = [HumanMessage(content=prompt)]
response = llm.invoke(messages)

In [61]:
print("\n===== VIDEO ANALYSIS =====\n")
print(response.content)


===== VIDEO ANALYSIS =====

# Video Analysis: "Every Real AI Project Starts Here: How to Create a Client Proposal"

## Concise Summary
- **Project continuation**: This is Part 2 of a series where the instructor builds on previous client requirement discussions to create a formal proposal document
- **Proposal fundamentals**: Covers what components should be included in an AI/software project proposal based on client needs identified in the previous session
- **Implementation roadmap**: Outlines the sequential process: Proposal → Sign-off → HLD/LLD → Architecture → System Access → Integration → Configuration
- **ERP integration focus**: Emphasizes connecting to client systems (SAP, ERP) to extract relevant data, documents, and purchase orders
- **Structured approach**: Demonstrates a professional methodology for translating client requirements into actionable project deliverables

## Main Topics/Themes
1. Proposal document structure and content requirements
2. Client requirement recap 